# Mixture-of-Depths：哪些 Token 值得进入昂贵层

**面试问题：token top-k、固定容量、旁路 residual 和路由可观测性怎样实现？**

## 回答主线

用一个可读任务先建立朴素 baseline，再从基础算子实现核心机制，输出中间状态、指标对照和失败修正。断言只在最后保护最关键的不变量；受控小数据用于解释机制，不冒充真实基础模型质量。

## 真实案例

一个长客服请求包含模板词、订单实体、金额、否定词和用户问题。计算预算只允许一半 token 进入昂贵 Transformer block。案例对比全量计算与固定容量 MoD，展示每个 token 的 router score、被选 token、旁路输出和计算节省；失败案例使用逐样本比例阈值导致容量超卖，说明 top-k 必须绑定物理预算。

### 输入预览：十二个具有不同信息量的 Token

In [1]:
import numpy as np  # 导入数组运算以实现 token 路由和残差块。

tokens = ["<用户>", "请", "不要", "重复", "扣款", "订单", "A7842", "金额", "899元", "已经", "支付", "怎么办"]  # 构造含否定、订单号和金额的真实客服请求。
features = np.array([[0.1, 0.1, 0.0], [0.1, 0.0, 0.1], [0.9, 0.6, 0.2], [0.5, 0.3, 0.2], [0.8, 0.4, 0.5], [0.5, 0.7, 0.4], [1.0, 0.9, 0.8], [0.4, 0.5, 0.3], [0.9, 0.8, 1.0], [0.2, 0.1, 0.2], [0.6, 0.4, 0.4], [0.7, 0.5, 0.6]], dtype=float)  # 用实体性、风险和任务相关性三个特征表示 token。
router_weight = np.array([0.45, 0.30, 0.25])  # 定义可解释的路由打分权重。
scores = features @ router_weight  # 计算每个 token 进入昂贵层的优先级。
print("token 路由输入：")  # 输出 token、三个特征和分数。
for token, feature, score in zip(tokens, features, scores):  # 逐 token 展示路由依据。
    print(f"{token:<8} features={feature} score={score:.3f}")  # 显示订单号、金额和否定词的高分。

token 路由输入：
<用户>     features=[0.1 0.1 0. ] score=0.075
请        features=[0.1 0.  0.1] score=0.070
不要       features=[0.9 0.6 0.2] score=0.635
重复       features=[0.5 0.3 0.2] score=0.365
扣款       features=[0.8 0.4 0.5] score=0.605
订单       features=[0.5 0.7 0.4] score=0.535
A7842    features=[1.  0.9 0.8] score=0.920
金额       features=[0.4 0.5 0.3] score=0.405
899元     features=[0.9 0.8 1. ] score=0.895
已经       features=[0.2 0.1 0.2] score=0.170
支付       features=[0.6 0.4 0.4] score=0.490
怎么办      features=[0.7 0.5 0.6] score=0.615


## Baseline 基线：所有 Token 都通过昂贵 Block

In [2]:
block_weight = np.array([[0.9, 0.1, -0.2], [0.2, 0.8, 0.1], [-0.1, 0.3, 0.7]])  # 定义教学版昂贵 block 的线性变换。

def expensive_block(hidden):  # 实现带非线性的 token block。
    return np.tanh(hidden @ block_weight)  # 对每个选中 token 执行相同昂贵变换。

full_output = features + expensive_block(features)  # 让全部十二个 token 执行 residual block。
full_compute = len(tokens)  # 用 block 调用 token 数衡量理论计算量。
print(f"全量基线执行 token 数={full_compute}")  # 展示 baseline 没有动态节省。
print("订单号 token 全量输出：", np.round(full_output[tokens.index("A7842")], 4))  # 展示一个关键信息 token 的具体 block 输出。
print("模板词 token 全量输出：", np.round(full_output[tokens.index("请")], 4))  # 展示低信息模板词也支付同样计算成本。

全量基线执行 token 数=12
订单号 token 全量输出： [1.7616 1.6857 1.2219]
模板词 token 全量输出： [0.1798 0.04   0.15  ]


### 核心实现：固定容量 Top-K 与 Residual 旁路

In [3]:
def mixture_of_depths(hidden, route_scores, capacity):  # 实现固定容量 token 路由与旁路 residual。
    stable_order = np.lexsort((np.arange(len(route_scores)), -route_scores))  # 按分数降序并用原位置稳定打破平局。
    selected = np.sort(stable_order[:capacity])  # 选择固定数量 token 并恢复原始因果顺序。
    output = hidden.copy()  # 未选 token 默认直接旁路保持 residual 身份映射。
    transformed = expensive_block(hidden[selected])  # 只对选中 token 执行昂贵 block。
    output[selected] = hidden[selected] + transformed  # 把 block 输出写回选中位置。
    mask = np.zeros(len(hidden), dtype=bool)  # 构造可观测的二值路由 mask。
    mask[selected] = True  # 标记真正进入 block 的 token。
    return output, mask, selected  # 返回完整序列输出、mask 和索引。

capacity = 6  # 把物理 block 容量固定为十二个 token 的一半。
mod_output, route_mask, selected_indices = mixture_of_depths(features, scores, capacity)  # 执行固定容量 MoD。
print("进入昂贵层的 token：", [tokens[index] for index in selected_indices])  # 展示被选择的否定、实体、金额和任务词。
print("旁路 token：", [token for token, selected in zip(tokens, route_mask) if not selected])  # 展示模板词直接沿 residual 传递。
print("路由 mask：", route_mask.astype(int).tolist())  # 展示位置保持和固定容量。

进入昂贵层的 token： ['不要', '扣款', '订单', 'A7842', '899元', '怎么办']
旁路 token： ['<用户>', '请', '重复', '金额', '已经', '支付']
路由 mask： [0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1]


## 结果解读：计算节省与关键信息保持

In [4]:
selected_error = float(np.max(np.abs(mod_output[route_mask] - full_output[route_mask])))  # 比较选中 token 与全量 block 的逐值误差。
bypass_change = float(np.max(np.abs(mod_output[~route_mask] - features[~route_mask])))  # 验证未选 token 保持 residual 旁路。
compute_saving = 1.0 - capacity / full_compute  # 计算理论 token-block 调用节省比例。
print("token      score  selected  output")  # 输出逐 token 路由结果表头。
for token, score, selected, output in zip(tokens, scores, route_mask, mod_output):  # 逐 token 展示选择和结果。
    print(f"{token:<8} {score:6.3f} {str(bool(selected)):<8} {np.round(output, 3)}")  # 让路由行为和输出直接可见。
print(f"理论 block token 计算节省={compute_saving:.1%}，选中路径误差={selected_error:.1e}，旁路变化={bypass_change:.1e}")  # 同时报告效率与正确性。

token      score  selected  output
<用户>      0.075 False    [0.1 0.1 0. ]
请         0.070 False    [0.1 0.  0.1]
不要        0.635 True     [1.621 1.158 0.22 ]
重复        0.365 False    [0.5 0.3 0.2]
扣款        0.605 True     [1.435 0.901 0.726]
订单        0.535 True     [1.001 1.323 0.645]
A7842     0.920 True     [1.762 1.686 1.222]
金额        0.405 False    [0.4 0.5 0.3]
899元      0.895 True     [1.601 1.574 1.537]
已经        0.170 False    [0.2 0.1 0.2]
支付        0.490 False    [0.6 0.4 0.4]
怎么办       0.615 True     [1.285 1.072 0.919]
理论 block token 计算节省=50.0%，选中路径误差=0.0e+00，旁路变化=0.0e+00


## 失败案例：阈值路由使 Batch 容量超卖

In [5]:
batch_scores = [scores, np.array([0.9, 0.88, 0.86, 0.84, 0.82, 0.80, 0.78, 0.76, 0.74, 0.72, 0.70, 0.68])]  # 构造第二个高信息密度请求。
threshold = 0.55  # 设置看似合理但不绑定物理容量的独立阈值。
threshold_counts = [int(np.sum(row >= threshold)) for row in batch_scores]  # 计算每个请求会路由多少 token。
physical_capacity = 12  # 假设整个 batch 的 kernel 只能处理十二个 token。
oversubscribed = sum(threshold_counts) > physical_capacity  # 判断独立阈值是否突破全局容量。
fixed_allocations = [6, 6]  # 使用每请求固定 Top-6 保证 batch 总量可预测。
print("阈值路由每请求选中数量：", threshold_counts, "总计=", sum(threshold_counts))  # 展示高密度样本导致容量不稳定。
print("物理容量：", physical_capacity, "是否超卖：", oversubscribed)  # 展示理论稀疏不等于 kernel 可调度。
print("固定 Top-K 分配：", fixed_allocations, "总计=", sum(fixed_allocations))  # 展示修正后的确定性容量。

阈值路由每请求选中数量： [5, 12] 总计= 17
物理容量： 12 是否超卖： True
固定 Top-K 分配： [6, 6] 总计= 12


### 生产边界

In [6]:
route_trace = {"sequence_tokens": len(tokens), "capacity": capacity, "selected_tokens": [tokens[index] for index in selected_indices], "router_version": "mod-router-r2", "compute_saving": compute_saving, "batch_capacity_policy": "fixed-top-k"}  # 构造可回放的路由 trace。
print("MoD trace：", route_trace)  # 展示排查质量回归和容量问题需要的字段。
print("生产替换点：真实 MoD 还需可训练 router、梯度估计、padding/packed 文档、分布式容量、kernel 紧凑布局和 decode 路由缓存。")  # 明确 NumPy top-k 与真实模型训练的差距。

MoD trace： {'sequence_tokens': 12, 'capacity': 6, 'selected_tokens': ['不要', '扣款', '订单', 'A7842', '899元', '怎么办'], 'router_version': 'mod-router-r2', 'compute_saving': 0.5, 'batch_capacity_policy': 'fixed-top-k'}
生产替换点：真实 MoD 还需可训练 router、梯度估计、padding/packed 文档、分布式容量、kernel 紧凑布局和 decode 路由缓存。


## 回归测试：只保护容量、顺序和旁路

In [7]:
assert int(route_mask.sum()) == capacity  # 验证每个请求严格使用固定 token 容量。
assert selected_indices.tolist() == sorted(selected_indices.tolist())  # 验证选中 token 写回时保持原始因果顺序。
assert selected_error < 1e-12  # 验证选中 token 与全量 block 使用相同计算路径。
assert bypass_change < 1e-12  # 验证未选 token 只经过 residual 身份旁路。
assert oversubscribed and sum(fixed_allocations) == physical_capacity  # 验证失败探针和修正容量都生效。
print("回归测试通过：固定容量、因果顺序、选中等价、旁路和 batch 超卖修正均成立。")  # 用少量断言总结 MoD 合同。

回归测试通过：固定容量、因果顺序、选中等价、旁路和 batch 超卖修正均成立。
